In [2]:
import requests
import time

# Open-Meteo API base URL
BASE_URL = "https://api.open-meteo.com/v1/forecast"

# Define grid parameters
LAT_STEP = 10   # Latitude step size (in degrees)
LON_STEP = 10   # Longitude step size (in degrees)
latitudes = range(-90, 91, LAT_STEP)  # From -90° to 90°
longitudes = range(-180, 181, LON_STEP)  # From -180° to 180°

# Function to fetch current temperature
def fetch_temperature(lat, lon):
    params = {
        "latitude": lat,
        "longitude": lon,
        "current_weather": True,
    }
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    data = response.json()
    return data["current_weather"]["temperature"]

# Global search
hottest = {"temperature": float("-inf"), "lat": None, "lon": None}
coldest = {"temperature": float("inf"), "lat": None, "lon": None}

for lat in latitudes:
    for lon in longitudes:
        try:
            temp = fetch_temperature(lat, lon)
            # Update hottest and coldest locations
            if temp > hottest["temperature"]:
                hottest = {"temperature": temp, "lat": lat, "lon": lon}
            if temp < coldest["temperature"]:
                coldest = {"temperature": temp, "lat": lat, "lon": lon}
        except Exception as e:
            print(f"Error fetching data for ({lat}, {lon}): {e}")
        time.sleep(1)  # Avoid hitting API rate limits

# Output results
print("Hottest Place:")
print(f"  Latitude: {hottest['lat']}, Longitude: {hottest['lon']}, Temperature: {hottest['temperature']}°C")

print("Coldest Place:")
print(f"  Latitude: {coldest['lat']}, Longitude: {coldest['lon']}, Temperature: {coldest['temperature']}°C")


KeyboardInterrupt: 

In [13]:
import concurrent.futures

In [1]:
import json
import requests
from datetime import datetime, timedelta
import concurrent.futures
import time

# Load the JSON file
file_path = "../Ecoregions_Coordinates.json"
with open(file_path, "r") as f:
    data = json.load(f)

# Flatten the data to extract all coordinates
coordinates = []
for region_type, regions in data.items():
    for region_name, region_data in regions.items():
        lat, lon = region_data["centroid"]
        coordinates.append({"region": region_name, "latitude": lat, "longitude": lon})

# Function to fetch historical data for the year 2023
def fetch_historical_average_temperature(lat, lon, start_date, end_date):
    BASE_URL = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "temperature_unit": "celsius",
        "daily": "temperature_2m_max",  # You can change this to weekly or monthly if needed
        "timezone": "UTC",
    }
    
    retries = 3  # Number of retries in case of error
    for _ in range(retries):
        try:
            response = requests.get(BASE_URL, params=params)
            
            # If rate limit exceeded (status code 429), wait before retrying
            if response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", 10))  # Default to 10 seconds
                print(f"Rate limit exceeded, retrying after {retry_after} seconds...")
                time.sleep(retry_after)
                continue
            
            response.raise_for_status()
            data = response.json()

            # Check if 'daily' and 'temperature_2m_max' are in the response and have valid data
            if "daily" in data and "temperature_2m_max" in data["daily"]:
                temperatures = data["daily"]["temperature_2m_max"]
                if temperatures:  # Ensure the list is not empty
                    return sum(temperatures) / len(temperatures)

        except requests.exceptions.RequestException as e:
            print(f"Error fetching data for coordinates ({lat}, {lon}): {e}")
            break  # Stop retrying if a different exception occurs

    # Return None if data is invalid or empty
    return None

# Function to process each coordinate and return the result
def process_coordinate(coord):
    avg_temp = fetch_historical_average_temperature(
        coord["latitude"], coord["longitude"], "2023-01-01", "2023-12-31"
    )
    return coord["region"], avg_temp

# Initialize variables for tracking the hottest place
hottest = {"region": None, "average_temperature": float("-inf")}

# Using ThreadPoolExecutor for parallel requests
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:  # Reduce max_workers to prevent hitting the rate limit
    # Submit tasks to the executor
    future_to_region = {executor.submit(process_coordinate, coord): coord for coord in coordinates}
    
    # Process the results as they complete
    for future in concurrent.futures.as_completed(future_to_region):
        region, avg_temp = future.result()
        
        if avg_temp is not None:  # Only update if temperature is valid
            # Update the hottest location if necessary
            if avg_temp > hottest["average_temperature"]:
                hottest = {"region": region, "average_temperature": avg_temp}
        else:
            print(f"No valid data for {region}")

# Print the results
if hottest["region"] is not None:
    print("Hottest Place on Average:")
    print(f"Region: {hottest['region']}, Average Temperature: {hottest['average_temperature']}°C")
else:
    print("No valid data found for any region.")


Rate limit exceeded, retrying after 10 seconds...
Rate limit exceeded, retrying after 10 seconds...
Rate limit exceeded, retrying after 10 seconds...
Rate limit exceeded, retrying after 10 seconds...
Rate limit exceeded, retrying after 10 seconds...


In [4]:
import json
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
import time

# Load the JSON file
file_path = "../Ecoregions_Coordinates.json"
with open(file_path, "r") as f:
    data = json.load(f)

# 
coordinates = []
#.items function returns a sort of list (dict_items data type that is non viewable that gives a iterable but not printable list of key-value pairs as tuples, so at top level i get biome and associated dict 
for region_type, regions in data.items():
    for region_name, region_data in regions.items(): #now I iterate over smaller subdictionary, regions, which contains dictonary w coordinate
        lat, lon = region_data["centroid"]#this gives the exact coordinates I require for the next step! 
        coordinates.append({"region": region_name, "latitude": lat, "longitude": lon})


# Function to fetch forecast data
def fetch_forecast_temperature(lat, lon):
    BASE_URL = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": "temperature_2m_max,temperature_2m_min",  # Max and Min temperatures
        "temperature_unit": "celsius",
        "timezone": "UTC",
    }
    
    retries = 3  # Number of retries for failed requests
    for attempt in range(retries):
        try:
            response = requests.get(BASE_URL, params=params, timeout=10)
            
            if response.status_code == 429:  # Handle rate-limiting
                retry_after = int(response.headers.get("Retry-After", 5))
                print(f"Rate limit exceeded. Retrying after {retry_after} seconds...")
                time.sleep(retry_after)
                continue
            
            response.raise_for_status()
            data = response.json()

            # Extract the max and min temperatures for the day
            if "daily" in data and "temperature_2m_max" in data["daily"] and "temperature_2m_min" in data["daily"]:
                max_temp = data["daily"]["temperature_2m_max"][0]
                min_temp = data["daily"]["temperature_2m_min"][0]
                return max_temp, min_temp

        except requests.exceptions.RequestException as e:
            print(f"Error fetching forecast data for coordinates ({lat}, {lon}): {e}")
            return None, None
    
    return None, None

# Batch processing function
def process_batch(batch):
    results = []
    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = {executor.submit(fetch_forecast_temperature, coord["latitude"], coord["longitude"]): coord for coord in batch}
        for future in futures:
            coord = futures[future]
            try:
                max_temp, min_temp = future.result()
                if max_temp is not None and min_temp is not None:
                    results.append({
                        "region": coord["region"],
                        "max_temperature": max_temp,
                        "min_temperature": min_temp,
                    })
            except Exception as e:
                print(f"Error processing {coord['region']}: {e}")
    return results

# Process all coordinates in batches
batch_size = 10
pause_time = 5  # Pause between batches
all_results = []

for i in range(0, len(coordinates), batch_size):
    batch = coordinates[i:i + batch_size]
    print(f"Processing batch {i // batch_size + 1}/{(len(coordinates) - 1) // batch_size + 1}...")
    batch_results = process_batch(batch)
    all_results.extend(batch_results)
    print(f"Batch {i // batch_size + 1} completed. Pausing for {pause_time} seconds...")
    time.sleep(pause_time)

# Convert results to a DataFrame
df = pd.DataFrame(all_results)

# Sort the DataFrame for hottest and coldest regions
hottest_places = df.sort_values(by="max_temperature", ascending=False).head(30)
coldest_places = df.sort_values(by="min_temperature", ascending=True).head(30)

# Display the results
print("Top 30 Hottest Places:")
print(hottest_places)
print("\nTop 30 Coldest Places:")
print(coldest_places)

# Save to CSV
hottest_places.to_csv("hottest_places.csv", index=False)
coldest_places.to_csv("coldest_places.csv", index=False)


Processing batch 1/85...
Batch 1 completed. Pausing for 5 seconds...
Processing batch 2/85...
Batch 2 completed. Pausing for 5 seconds...
Processing batch 3/85...
Error fetching forecast data for coordinates (38.617963544595284, 105.86100349816759): HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Error fetching forecast data for coordinates (35.395588965671976, 0.9022264495101009): HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Error fetching forecast data for coordinates (29.194900820558537, 100.27070572591626): HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Error fetching forecast data for coordinates (29.070190792035778, 97.74017818214652): HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
Error fetching forecast data for coordinates (29.46980439531501, 94.68759622678401): HTTPSConnectionPool(host='api.open-me

KeyboardInterrupt: 

In [13]:
import nest_asyncio
import asyncio
import aiohttp
import pandas as pd
import json

nest_asyncio.apply()

# Load JSON file
file_path = "../Ecoregions_Coordinates.json"
with open(file_path, "r") as f:
    data = json.load(f)

# Flatten data

# .items function returns a sort of list (dict_items data type that is non viewable that gives a iterable but not printable list of key-value pairs as tuples, so at top level i get biome and associated dict 

coordinates = [
    {"region": region_name, "latitude": region_data["centroid"][0], "longitude": region_data["centroid"][1]}
    for region_type, regions in data.items()
    for region_name, region_data in regions.items()
]

# Fetch data, note to future self coord is just one entry in coordinates list of dictionaries js renamed it so as to avoid any confusion!
async def fetch_forecast_temperature(session, coord):
    BASE_URL = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": coord["latitude"],
        "longitude": coord["longitude"],
        "daily": "temperature_2m_max,temperature_2m_min",
        "temperature_unit": "celsius",
        "timezone": "UTC",
    }

    retries = 3
    for attempt in range(retries):
        try:
            async with session.get(BASE_URL, params=params, timeout=15) as response:
                if response.status == 429:
                    retry_after = int(response.headers.get("Retry-After", 5))
                    await asyncio.sleep(retry_after)
                    continue
                response.raise_for_status()
                data = await response.json()
                if "daily" in data and "temperature_2m_max" in data["daily"]:
                    return {
                        "region": coord["region"],
                        "max_temperature": data["daily"]["temperature_2m_max"][0],
                        "min_temperature": data["daily"]["temperature_2m_min"][0],
                    }
        except Exception as e:
            print(f"Error for {coord['region']}: {e}")
    return None

# Process all coordinates
async def process_all_coordinates():
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_forecast_temperature(session, coord) for coord in coordinates]
        results = await asyncio.gather(*tasks, return_exceptions=True)
    return [res for res in results if res]

# Main function
async def main():
    results = await process_all_coordinates()
    df = pd.DataFrame(results)
    hottest = df.sort_values(by="max_temperature", ascending=False).head(30)
    coldest = df.sort_values(by="min_temperature").head(30)
    hottest.to_csv("hottest_places.csv", index=False)
    coldest.to_csv("coldest_places.csv", index=False)

# Run the async function
await main()


In [11]:
import nest_asyncio
import asyncio
import aiohttp
import pandas as pd
import json

nest_asyncio.apply()

# Load JSON file
file_path = "../Ecoregions_Coordinates.json"
with open(file_path, "r") as f:
    data = json.load(f)

# Flatten data
coordinates = [
    {"region": region_name, "latitude": region_data["centroid"][0], "longitude": region_data["centroid"][1]}
    for region_type, regions in data.items()
    for region_name, region_data in regions.items()
]

# Fetch historical temperature data
async def fetch_historical_temperature(session, coord):
    BASE_URL = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": coord["latitude"],
        "longitude": coord["longitude"],
        "daily": "temperature_2m_max,temperature_2m_min",
        "temperature_unit": "celsius",
        "timezone": "UTC",
        "start_date": "2023-01-01",
        "end_date": "2023-12-31",
    }

    retries = 3
    for attempt in range(retries):
        try:
            async with session.get(BASE_URL, params=params) as response:
                if response.status == 429:  # Handle rate-limiting
                    retry_after = int(response.headers.get("Retry-After", 5))
                    print(f"Rate limit exceeded. Retrying after {retry_after} seconds...")
                    await asyncio.sleep(retry_after)
                    continue

                response.raise_for_status()
                data = await response.json()

                # Extract daily max and min temperatures
                if "daily" in data and "temperature_2m_max" in data["daily"] and "temperature_2m_min" in data["daily"]:
                    max_temps = [temp for temp in data["daily"]["temperature_2m_max"] if temp is not None]
                    min_temps = [temp for temp in data["daily"]["temperature_2m_min"] if temp is not None]

                    if max_temps and min_temps:  # Ensure lists are not empty
                        avg_max_temp = sum(max_temps) / len(max_temps)
                        avg_min_temp = sum(min_temps) / len(min_temps)
                        return {
                            "region": coord["region"],
                            "average_max_temperature": avg_max_temp,
                            "average_min_temperature": avg_min_temp,
                        }
        except aiohttp.ClientError as e:
            print(f"Error fetching data for {coord['region']}: {e}")
            return {
                "region": coord["region"],
                "average_max_temperature": None,
                "average_min_temperature": None,
            }

    return {
        "region": coord["region"],
        "average_max_temperature": None,
        "average_min_temperature": None,
    }

# Main function to process all coordinates asynchronously
async def process_coordinates(coordinates):
    results = []
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_historical_temperature(session, coord) for coord in coordinates]
        for task in asyncio.as_completed(tasks):
            result = await task
            if result:
                results.append(result)
    return results

# Run the asyncio loop
all_results = asyncio.run(process_coordinates(coordinates))

# Convert results to a DataFrame
df = pd.DataFrame(all_results)

# Sort to find the hottest and coldest places
hottest_places = df.sort_values(by="average_max_temperature", ascending=False).head(30)
coldest_places = df.sort_values(by="average_min_temperature", ascending=True).head(30)

# Display the results
print("Top 30 Hottest Places:")
print(hottest_places)
print("\nTop 30 Coldest Places:")
print(coldest_places)

# Save to CSV
hottest_places.to_csv("hottest_places_averages.csv", index=False)
coldest_places.to_csv("coldest_places_averages.csv", index=False)
hottest_places

Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded. Retrying after 5 seconds...
Rate limit exceeded.

,region,average_max_temperature,average_min_temperature
0,Carpentaria tropical savanna,None,None
1,Iquitos várzea,None,None
2,Sumba deciduous forests,None,None
3,Aldabra Island xeric scrub,None,None
4,South Victoria Land tundra,None,None
5,Central American Atlantic moist forests,None,None
6,Southern Vietnam lowland dry forests,None,None
7,Azerbaijan shrub desert and steppe,None,None
8,Puget lowland forests,None,None
9,Mediterranean dry woodlands and steppe,None,None


In [36]:
import pandas as pd
filepath = 'coldest_places.csv'
df_coldest_places=pd.read_csv(filepath)


# Drop the 'max_temperature' column
df_coldest_places = df_coldest_places.drop(columns=['max_temperature'])

coordinates_df = pd.DataFrame(coordinates)
df_coldest_places = pd.merge(df_coldest_places, coordinates_df, on="region", how="left")
df_coldest_places = df_coldest_places.rename(columns={'min_temperature': 'temperature'})
print(df_coldest_places)

                                               region  temperature   latitude  \
0                     Great Lakes Basin desert steppe        -32.2  48.741138   
1                           Sayan Intermontane steppe        -26.8  51.343495   
2                                  Amur meadow steppe        -25.4  48.718306   
3                 Western Siberian hemiboreal forests        -25.3  56.340969   
4                                       Kazakh steppe        -24.8  51.983704   
5                            Hindu Kush alpine meadow        -24.6  36.284239   
6        Da Hinggan-Dzhagdy Mountains conifer forests        -24.6  51.902679   
7                         New England-Acadian forests        -24.5  45.499181   
8                Gulf of St. Lawrence lowland forests        -24.5  46.493085   
9                                Kazakh upland steppe        -24.5  50.285132   
10                               Kazakh forest steppe        -24.4  54.416075   
11                          

In [35]:
import pandas as pd
filepath = 'hottest_places.csv'
df_hottest_places=pd.read_csv(filepath)


# Drop the 'max_temperature' column
df_hottest_places = df_hottest_places.drop(columns=['min_temperature'])

coordinates_df = pd.DataFrame(coordinates)
df_hottest_places = pd.merge(df_hottest_places, coordinates_df, on="region", how="left")
df_hottest_places = df_hottest_places.rename(columns={'max_temperature': 'temperature'})
print(df_hottest_places)

                                               region  temperature   latitude  \
0                                       Gibson desert         45.7 -24.346783   
1                                      Simpson desert         45.1 -26.274999   
2                           Great Sandy-Tanami desert         44.6 -21.445277   
3                  Eastern Australia mulga shrublands         44.3 -27.884564   
4                          Central Ranges xeric scrub         43.1 -24.552720   
5                                    Esperance mallee         41.6 -33.291211   
6                 Western Australian Mulga shrublands         41.2 -26.602173   
7                         Southwest Australia savanna         40.4 -29.944534   
8                                Coolgardie woodlands         40.1 -31.384561   
9                                  Pilbara shrublands         38.9 -21.914966   
10                            Sudd flooded grasslands         38.5   7.826086   
11                          

In [31]:
import folium

# Initialize a base map (centered at a midpoint between the hottest and coldest locations)
m = folium.Map(location=[df_hottest_places['latitude'].mean(), df_hottest_places['longitude'].mean()], zoom_start=2)

# Add markers for hottest places
for _, row in df_hottest_places.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        icon=folium.Icon(color='red', icon='cloud')
    ).add_to(m)

# Add markers for coldest places
for _, row in df_coldest_places.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        icon=folium.Icon(color='blue', icon='cloud')
    ).add_to(m)

# Save or display the map
m.save("temperature_map.html")
m


In [37]:
import folium
from branca.colormap import linear
import pandas as pd





# Initialize the base map centered between hottest and coldest places
m = folium.Map(location=[(df_hottest_places['latitude'].mean() + df_coldest_places['latitude'].mean()) / 2,
                         (df_hottest_places['longitude'].mean() + df_coldest_places['longitude'].mean()) / 2], zoom_start=2)

# Create color scales for hot and cold places
hot_colormap = linear.Reds_09.scale(df_hottest_places['temperature'].min(), df_hottest_places['temperature'].max())
cold_colormap = linear.Blues_09.scale(df_coldest_places['temperature'].min(), df_coldest_places['temperature'].max())

# Add markers for hottest places with gradient colors
for idx, row in df_hottest_places.iterrows():
    rank = idx + 1
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Rank: {rank}<br>Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        icon=folium.Icon(color='red', icon='cloud'),
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)",
    ).add_to(m)
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=10,
        color=hot_colormap(row['temperature']),
        fill=True,
        fill_opacity=0.8,
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)"
    ).add_to(m)

# Add markers for coldest places with gradient colors
for idx, row in df_coldest_places.iterrows():
    rank = idx + 1
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Rank: {rank}<br>Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        icon=folium.Icon(color='blue', icon='cloud'),
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)",
    ).add_to(m)
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=10,
        color=cold_colormap(row['temperature']),
        fill=True,
        fill_opacity=0.8,
        tooltip=f"Rank {rank}: {row['region']} ({row['temperature']}°C)"
    ).add_to(m)

# Add legends to the map
hot_colormap.caption = 'Temperature Scale (Hottest Places)'
cold_colormap.caption = 'Temperature Scale (Coldest Places)'
hot_colormap.add_to(m)
cold_colormap.add_to(m)

# Save or display the map
m.save("temperature_ranked_map.html")
m


In [30]:
import folium
from folium.plugins import HeatMap
from branca.colormap import linear
import pandas as pd



# Initialize the base map centered at an average location
m = folium.Map(location=[(df_hottest_places['latitude'].mean() + df_coldest_places['latitude'].mean()) / 2,
                         (df_hottest_places['longitude'].mean() + df_coldest_places['longitude'].mean()) / 2], zoom_start=2)

# Create a color scale for temperatures
colormap = linear.RdYlBu_11.scale(df_coldest_places['temperature'].min(), df_hottest_places['temperature'].max())

# Function to map temperature to color intensity
def get_marker_color(temp, temp_min, temp_max, color_scale):
    # Map temperature to a hex color
    hex_color = color_scale(temp)
    return hex_color

# Add markers for hottest places
for _, row in df_hottest_places.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=8,
        popup=f"Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                               df_hottest_places['temperature'].max(), colormap),
        fill=True,
        fill_color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                                    df_hottest_places['temperature'].max(), colormap),
        fill_opacity=0.8
    ).add_to(m)

# Add markers for coldest places
for _, row in df_coldest_places.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=2,
        popup=f"Region: {row['region']}<br>Temperature: {row['temperature']}°C",
        color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                               df_hottest_places['temperature'].max(), colormap),
        fill=True,
        fill_color=get_marker_color(row['temperature'], df_coldest_places['temperature'].min(), 
                                    df_hottest_places['temperature'].max(), colormap),
        fill_opacity=0.8
    ).add_to(m)

# Add color scale legend to map
colormap.caption = 'Temperature Scale (°C)'
colormap.add_to(m)

# Add HeatMap for temperature intensity
heat_data = [[row['latitude'], row['longitude'], row['temperature']] 
             for _, row in pd.concat([df_hottest_places, df_coldest_places]).iterrows()]
HeatMap(heat_data).add_to(m)

# Save or display the map
m.save("temperature_map_with_scale.html")
m
